### 路徑設定
- **模組**沿用 `rebuild_data/`（schema / parse）
- **資料根目錄**獨立於 `rebuild.ipynb` 的 `rebuild_root`
- zip **不解壓落地**：`read_tej_zip` 只讀進記憶體，減少 SSD 寫入
- 本 notebook 目前只做**載入**，處理流程後續再寫

In [1]:
import sys
import os
import zipfile
import importlib

import polars as pl

# Python 模組：rebuild_data（與全量重建共用）
modules_root = "/home/newt/quant_project/rebuild_data"
if modules_root not in sys.path:
    sys.path.insert(0, modules_root)

# 每日增量資料根目錄（獨立，勿與 rebuild_root 混用）
update_root = "/mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily"
zip_dir = os.path.join(update_root, "daily_add")     # TEJ 下載的 zip（直接讀，不解壓）
save_path = os.path.join(update_root, "daily_save")  # 輸出（後續用）

os.makedirs(save_path, exist_ok=True)
print("modules_root:", modules_root)
print("update_root :", update_root)
print("zip_dir     :", zip_dir)
print("save_path   :", save_path)

modules_root: /home/newt/quant_project/rebuild_data
update_root : /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily
zip_dir     : /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily/daily_add
save_path   : /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily/daily_save


In [2]:
from parse import read_tej_txt, read_tej_zip


def reload_modules(module_list):
    for mod in module_list:
        importlib.reload(mod)


def daily_zip(name: str) -> str:
    """zip_dir 內 <name>.zip；不存在則 raise。"""
    path = os.path.join(zip_dir, f"{name}.zip")
    if not os.path.isfile(path):
        raise FileNotFoundError(f"找不到 zip：{path}")
    return path


def try_daily_zip(name: str) -> str | None:
    path = os.path.join(zip_dir, f"{name}.zip")
    return path if os.path.isfile(path) else None


def extract_zips_inplace(zip_dir: str) -> list[str]:
    """備援：把 zip 內 .txt 解到 zip 同目錄（不另開 daily_work）。

    日常請用 read_tej_zip（記憶體）；只有外部工具一定要實體 txt 時才呼叫。
    """
    stems = []
    for name in sorted(os.listdir(zip_dir)):
        if not name.endswith(".zip"):
            continue
        zpath = os.path.join(zip_dir, name)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(zip_dir)
        stems.append(name[:-4])
        print(f"extracted inplace: {name} → {zip_dir}")
    return stems

## 資料載入（僅讀取，不處理）
一律 `read_tej_zip(daily_zip(...))`，不解壓到 SSD。
`calendar` / `company_info` 若每日 zip 沒有則 skip。

### Trading_Halt

In [3]:
import Trading_Halt

trading_halt_df = read_tej_zip(
    daily_zip("trading_halt"),
    Trading_Halt.TRADING_HALT_SCHEMA,
).select(Trading_Halt.keep_col)
print("trading_halt_df:", trading_halt_df.shape)
trading_halt_df.head()

trading_halt_df: (0, 4)


symbol,start_halt,resume_trading_date,reason
str,date,date,str


### OHLCV（Daily_Unadjusted_Price）

In [4]:
import OHLCV
from OHLCV import OHLCV_SCHEMA

OHLCV_df = read_tej_zip(
    daily_zip("Daily_Unadjusted_Price"),
    OHLCV_SCHEMA,
).select(OHLCV.keep_col)
print("OHLCV_df:", OHLCV_df.shape)
OHLCV_df = OHLCV_df.filter(OHLCV_df["market"].is_in(["TSE", "OTC"]))
OHLCV_df = OHLCV_df.with_columns([
    pl.lit(0).alias("untradable"),
    pl.lit(1).alias("adj_factor"),
])
OHLCV_df.head()

OHLCV_df: (2313, 22)


symbol,date,open,high,low,close,volume,amount,return_day,turnover,market_value_million,log_return_day,numtrades,limit_status,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market,untradable,adj_factor
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,str,str,str,str,i32,i32
"""1101""",2026-06-01,24.4,24.7,24.2,24.55,35557.0,869509.0,0.6148,0.4726,184694.0,0.6129,10973.0,null,2.0492,24.55,27.0,22.1,null,null,null,"""TSE""",0,1
"""1102""",2026-06-01,34.0,34.15,33.65,33.8,16350.0,554484.0,-0.5882,0.461,119874.0,-0.59,8219.0,null,1.4706,33.8,37.15,30.45,null,null,null,"""TSE""",0,1
"""1103""",2026-06-01,13.45,13.45,13.0,13.3,606.0,8024.0,-1.1152,0.0728,11077.0,-1.1215,489.0,null,3.3457,13.3,14.6,12.0,null,null,null,"""TSE""",0,1
"""1104""",2026-06-01,27.65,27.95,27.6,27.9,969.0,26998.0,1.6393,0.1411,19158.0,1.6261,711.0,null,1.275,27.9,30.65,25.15,null,null,null,"""TSE""",0,1
"""1108""",2026-06-01,13.75,13.85,13.65,13.8,341.0,4701.0,0.7299,0.0843,5585.0,0.7273,162.0,null,1.4599,13.8,15.15,12.45,null,null,null,"""TSE""",0,1


### Cash_Dividends

In [ ]:
import Cash_Dividends

cash_dividend_df = read_tej_zip(
    daily_zip("Cash_Dividends"),
    Cash_Dividends.CASH_DIVIDENDS_SCHEMA,
).select(Cash_Dividends.keep_col)

print("cash_dividend_df:", cash_dividend_df.shape)
cash_dividend_df.head()

cash_dividend_df: (5, 5)


symbol,date,dividend_per_share,prev_close,ex_div_ref_price
str,date,f64,f64,f64
"""2612""",2026-06-01,2.2,57.7,55.5
"""4527""",2026-06-01,3.0,45.95,42.95
"""6176""",2026-06-01,3.5,106.5,103.0
"""6909""",2026-06-01,1.0001428,53.5,52.49
"""8924""",2026-06-01,4.3,64.5,60.2


### Corporate_Actions

In [ ]:
import Corporate_Actions
from Corporate_Actions import CORPORATE_ACTIONS_SCHEMA

ex_rights_df = read_tej_zip(
    daily_zip("Corporate_Actions"),
    CORPORATE_ACTIONS_SCHEMA,
)
print("ex_rights_df:", ex_rights_df.shape)
ex_rights_df.head()

### institutional_return

In [ ]:
import institutional_return

reload_modules([institutional_return])
institutional_return_df = read_tej_zip(
    daily_zip("institutional_return"),
    institutional_return.INSTITUTIONAL_RETURN_SCHEMA,
)
print("institutional_return_df:", institutional_return_df.shape)
institutional_return_df.head()

### institutional_net_flow

In [ ]:
import institutional_net_flow

reload_modules([institutional_net_flow])
institutional_net_flow_df = read_tej_zip(
    daily_zip("institutional_net_flow"),
    institutional_net_flow.INSTITUTIONAL_NET_FLOW_SCHEMA,
).select(institutional_net_flow.keep_col)
print("institutional_net_flow_df:", institutional_net_flow_df.shape)
institutional_net_flow_df.head()

### Credit_Transactions

In [ ]:
import Credit_Transactions

reload_modules([Credit_Transactions])
Credit_Transaction_df = read_tej_zip(
    daily_zip("Credit_Transactions"),
    Credit_Transactions.CREDIT_TRANSACTIONS_SCHEMA,
)
print("Credit_Transaction_df:", Credit_Transaction_df.shape)
Credit_Transaction_df.head()

### Day_Trading_Stats

In [ ]:
import Day_Trading_Stats

reload_modules([Day_Trading_Stats])
Day_Trading_Stats_df = read_tej_zip(
    daily_zip("Day_Trading_Stats"),
    Day_Trading_Stats.DAY_TRADING_STATS_SCHEMA,
)
print("Day_Trading_Stats_df:", Day_Trading_Stats_df.shape)
Day_Trading_Stats_df.head()

### Securities_Lending
rebuild 標為有問題暫不用；每日 zip 有檔，先載入原始資料備用。

In [ ]:
import Securities_Lending

reload_modules([Securities_Lending])
Securities_Lending_df = read_tej_zip(
    daily_zip("Securities_Lending"),
    Securities_Lending.SECURITIES_LENDING_SCHEMA,
)
print("Securities_Lending_df:", Securities_Lending_df.shape)
Securities_Lending_df.head()

### share_distribution

In [ ]:
import share_distribution

reload_modules([share_distribution])
share_distribution_df = read_tej_zip(
    daily_zip("share_distribution"),
    share_distribution.SHARE_DISTRIBUTION_SCHEMA,
)
print("share_distribution_df:", share_distribution_df.shape)
share_distribution_df.head()

### Company_Info

In [ ]:
import Company_Info

reload_modules([Company_Info])
_ci_zip = try_daily_zip("company_info")
if _ci_zip:
    company_info_df = read_tej_zip(_ci_zip)
    print("company_info_df:", company_info_df.shape)
    company_info_df.head()
else:
    company_info_df = None
    print("skip company_info：zip_dir 無 company_info.zip")

## 載入摘要

In [ ]:
_loaded = {
    "calendar_df": calendar_df,
    "trading_halt_df": trading_halt_df,
    "OHLCV_df": OHLCV_df,
    "cash_dividend_df": cash_dividend_df,
    "ex_rights_df": ex_rights_df,
    "institutional_return_df": institutional_return_df,
    "institutional_net_flow_df": institutional_net_flow_df,
    "Credit_Transaction_df": Credit_Transaction_df,
    "Day_Trading_Stats_df": Day_Trading_Stats_df,
    "Securities_Lending_df": Securities_Lending_df,
    "share_distribution_df": share_distribution_df,
    "company_info_df": company_info_df,
}
for name, df in _loaded.items():
    if df is None:
        print(f"{name:30s}  SKIP")
    else:
        print(f"{name:30s}  {df.shape[0]:>8} rows × {df.shape[1]} cols")

## 處理流程（待寫）
後續在此接 halt / 復權 / 籌碼合併等，對齊 `rebuild.ipynb` 邏輯但改為每日增量。